Load OAI-PMH Data Providers

In [29]:
from os import getenv
from sqlalchemy import create_engine, Column, Integer, String, Float, JSON, DateTime
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker, create_async_engine
from sqlalchemy.orm import declarative_base
import httpx
from datetime import datetime, UTC
from bs4 import BeautifulSoup
from xml.dom import minidom

engine = create_async_engine(getenv("DATABASE_URL"), echo=True, future=True)

AsyncSessionLocal = async_sessionmaker(
    engine,
    class_=AsyncSession,
    expire_on_commit=False,
    autocommit=False,
    autoflush=False,
)

Base = declarative_base()

session = AsyncSessionLocal()

Define Model OAI_PMH

In [10]:
class OAI_PMH(Base):
    __tablename__ = 'OAI_PMH'
    id = Column(Integer, primary_key=True)
    repository_name = Column(String(500),  nullable=False)
    url = Column(String(500),  nullable=False)
    namespace_identifier = Column(String(150),  nullable=False)
    

Define Model ROAR

In [11]:
class ROAR(Base):
    __tablename__ = 'ROAR'
    id = Column(Integer, primary_key=True)
    repository_name = Column(String(500),  nullable=False)
    home_page = Column(String(500),  nullable=False)
    oai_pmh = Column(String(150),  nullable=True)
    

Create tables

In [13]:
async with engine.begin() as conn:
    await conn.run_sync(BASE.metadata.create_all)

2026-02-09 14:24:33,841 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-02-09 14:24:33,842 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-09 14:24:34,137 INFO sqlalchemy.engine.Engine select current_schema()
2026-02-09 14:24:34,139 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-09 14:24:34,531 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-02-09 14:24:34,534 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-09 14:24:34,825 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-09 14:24:34,827 INFO sqlalchemy.engine.Engine COMMIT


Create records model OAI_PMH

In [ ]:
## URL deprecated

URL = 'https://www.openarchives.org/Register/BrowseSites'
page = httpx.get(URL)
soup = BeautifulSoup(page.content, 'html.parser')
records = []
for i, item in enumerate(soup('tr')[1:]):
    th_list = item.find_all('td')
    if 3 < len(th_list) and th_list[2].text != ('' or ' '):
        records.append(OAI_PMH(th_list[2].text, th_list[3].text, th_list[4].text))

In [37]:
print(len(records))

0


Insert data OAI_PMH

In [18]:
session.add_all(records)
await session.commit()

Create records model ROAR

In [33]:
mydoc = minidom.parse('data/rawlist.xml')
# records_roar = [ROAR(i.getElementsByTagName('title')[0].firstChild.nodeValue, i.getElementsByTagName('home_page')[0].firstChild.nodeValue, i.getElementsByTagName('oai_pmh')[0].getElementsByTagName('item')[0].firstChild.nodeValue) for i in mydoc.getElementsByTagName('eprint')]
roar_list = []
for i in mydoc.getElementsByTagName('eprint'):
    title = i.getElementsByTagName('title')
    if title: 
        oai_pmh = i.getElementsByTagName('oai_pmh')
        roar_list.append(
            ROAR(**{
                'repository_name': i.getElementsByTagName('title')[0].firstChild.nodeValue, 
                'home_page': i.getElementsByTagName('home_page')[0].firstChild.nodeValue, 
                'oai_pmh': oai_pmh[0].getElementsByTagName('item')[0].firstChild.nodeValue if oai_pmh else None
            })
        )

Insert data ROAR

In [35]:
session.add_all(roar_list)
await session.commit()

2026-02-09 14:36:23,058 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-09 14:36:23,091 INFO sqlalchemy.engine.Engine INSERT INTO "ROAR" (repository_name, home_page, oai_pmh) SELECT p0::VARCHAR, p1::VARCHAR, p2::VARCHAR FROM (VALUES ($1::VARCHAR, $2::VARCHAR, $3::VARCHAR, 0), ($4::VARCHAR, $5::VARCHAR, $6::VARCHAR, 1), ($7::VARCHAR, $8::VARCHAR, $9::VARCHAR, 2), ($1 ... 53644 characters truncated ... 9)) AS imp_sen(p0, p1, p2, sen_counter) ORDER BY sen_counter RETURNING "ROAR".id, "ROAR".id AS id__1
2026-02-09 14:36:23,092 INFO sqlalchemy.engine.Engine [cached since 333.1s ago (insertmanyvalues) 1/5 (ordered)] ('@RCHIVESIC ', 'http://archivesic.ccsd.cnrs.fr/', 'http://archivesic.ccsd.cnrs.fr/oai/oai.php', 'Australian Agriculture and Natural Resource Online: AANRO  ', 'http://www.aanro.net/HOME.html', None, 'Abertay Research Collections', 'https://repository.abertay.ac.uk/jspui/', 'http://repository.abertay.ac.uk/oai/request', 'Academic Archive On-line (Jönköping University, Swed